# Megaminx breakthrough stack — direct `git clone` Colab notebook

Этот ноутбук рассчитан на запуск через **Run all** в Google Colab.

Что он делает:
1. Устанавливает системные зависимости.
2. Клонирует репозиторий напрямую через `git clone https://github.com/visualcomments/agents_4_puzzles`.
3. Устанавливает Python-зависимости.
4. Проверяет, что все breakthrough-патчи уже лежат **внутри репозитория**.
5. Запускает `competitions/cayley-py-megaminx/turnkey_real_external_run.py`.
6. Собирает итоговый zip с артефактами.

Важно: ноутбук не применяет внешний patch bundle. Он ожидает, что нужные файлы уже доступны
в самом клонируемом репозитории.


In [ ]:
import json, os, pathlib, shlex, subprocess, sys

REPO_URL = "https://github.com/visualcomments/agents_4_puzzles"
REPO_REF = ""  # empty = default branch
REPO_DIR = pathlib.Path('/content/agents_4_puzzles')
WORK_DIR = 'competitions/cayley-py-megaminx/runs/turnkey_real_external'
ZIP_OUT = pathlib.Path('/content/megaminx_real_external_runall_from_gitclone.zip')
SKIP_EXTERNAL = False
SKIP_HARD_ROW = False
TOP_K = 64
LIGHT_TIME_BUDGET_PER_ROW = 0.02
AGGRESSIVE_TIME_BUDGET_PER_ROW = 0.05

def run(cmd, cwd=None, check=True, env=None, timeout=None):
    if isinstance(cmd, str):
        cmd = shlex.split(cmd)
    print('+', ' '.join(shlex.quote(str(x)) for x in cmd))
    return subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        check=check,
        env=env,
        timeout=timeout,
    )


In [ ]:
run(['apt-get', '-qq', 'update'])
run([
    'apt-get', '-qq', 'install', '-y',
    'git', 'build-essential', 'python3-venv', 'cargo', 'rustc',
    'openjdk-17-jre-headless', 'jq', 'zip'
])
run([sys.executable, '-m', 'pip', 'install', '-U', 'pip', 'setuptools', 'wheel', 'nbformat'])


In [ ]:
if REPO_DIR.exists():
    run(['rm', '-rf', str(REPO_DIR)])
clone_cmd = ['git', 'clone']
if REPO_REF:
    clone_cmd += ['--depth', '1', '--branch', REPO_REF]
clone_cmd += [REPO_URL, str(REPO_DIR)]
run(clone_cmd)
try:
    run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements-full.txt')])
except subprocess.CalledProcessError:
    run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements-min.txt')])
try:
    run([sys.executable, '-m', 'pip', 'install', 'git+https://github.com/cayleypy/cayleypy.git'])
except subprocess.CalledProcessError:
    run([sys.executable, '-m', 'pip', 'install', 'cayleypy'], check=False)


In [ ]:
        required = [
            REPO_DIR / 'competitions/cayley-py-megaminx/turnkey_real_external_run.py',
            REPO_DIR / 'competitions/cayley-py-megaminx/external_adapter_lane.py',
            REPO_DIR / 'competitions/cayley-py-megaminx/portfolio_orchestrator.py',
            REPO_DIR / 'competitions/cayley-py-megaminx/prompt_population_runner.py',
            REPO_DIR / 'competitions/cayley-py-megaminx/hard_row_routed_search.py',
            REPO_DIR / 'competitions/cayley-py-megaminx/row_scoreboard.py',
            REPO_DIR / 'competitions/cayley-py-megaminx/shadow_splits.json',
            REPO_DIR / 'competitions/cayley-py-megaminx/external_solver_adapters/manifest_real_odder_megaminxolver.json',
            REPO_DIR / 'competitions/cayley-py-megaminx/external_solver_adapters/manifest_real_sevilze_llminxsolver_cmp.json',
            REPO_DIR / 'competitions/cayley-py-megaminx/external_solver_adapters/manifest_real_abgolev_astar.json',
        ]
        missing = [str(p) for p in required if not p.exists()]
        if missing:
            raise FileNotFoundError('Missing required repository-integrated files:
' + '
'.join(missing))
        print('All required repository-integrated patches are present.')


In [ ]:
cmd = [
    sys.executable,
    'competitions/cayley-py-megaminx/turnkey_real_external_run.py',
    '--work-dir', WORK_DIR,
    '--zip-out', str(ZIP_OUT),
    '--top-k', str(TOP_K),
    '--light-time-budget-per-row', str(LIGHT_TIME_BUDGET_PER_ROW),
    '--aggressive-time-budget-per-row', str(AGGRESSIVE_TIME_BUDGET_PER_ROW),
]
if SKIP_EXTERNAL:
    cmd.append('--skip-external')
if SKIP_HARD_ROW:
    cmd.append('--skip-hard-row')
run(cmd, cwd=REPO_DIR)


In [ ]:
summary_path = REPO_DIR / WORK_DIR / 'turnkey_run_summary.json'
if summary_path.exists():
    print(summary_path.read_text(encoding='utf-8'))
else:
    print('Summary file not found:', summary_path)
print('ZIP_OUT:', ZIP_OUT)
print('ZIP exists:', ZIP_OUT.exists())


In [ ]:
# Optional download helper for Colab
try:
    from google.colab import files  # type: ignore
    if ZIP_OUT.exists():
        print('Ready for download:', ZIP_OUT)
        # files.download(str(ZIP_OUT))
except Exception:
    pass
